# Lektion 10 - KI-Agenten in Produktion

In dieser Lektion lernen Sie **Produktionsmuster** für KI-Agenten unter Verwendung des Microsoft Agent Framework (Python). Wir behandeln:

- **Beobachtbarkeit** — Hinzufügen von Zeitmessung und Protokollierung zu Agenteninteraktionen
- **Bewertung** — Einsatz eines Bewertungsagenten zur Bewertung der Antwortqualität
- **Kostenmanagement** — Strategien zur Token-Optimierung und Modellauswahl

Das Szenario ist ein **Reiseberater**, der Nutzern beim Planen von Reisen hilft, mit Überwachung und Auswertung obendrauf.


## Einrichtung


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Produktionsüberlegungen

Die Überführung von KI-Agenten von Prototypen in die Produktion erfordert sorgfältige Beachtung dreier Säulen:

1. **Beobachtbarkeit** — Sie benötigen Einblick darin, was der Agent tut, wie lange er dafür braucht und welche Tools er aufruft. Ohne Tracing und Logging ist die Fehlerbehebung bei Produktionsproblemen nahezu unmöglich.

2. **Evaluierung** — Automatisierte Qualitätsprüfungen stellen sicher, dass die Antworten des Agenten über die Zeit hinweg genau, vollständig und hilfreich bleiben. Ein Bewertungsagent kann Antworten gegen definierte Kriterien bewerten.

3. **Kostenmanagement** — Token-Nutzung wirkt sich direkt auf die Kosten aus. Strategien wie Prompt-Optimierung, Modellauswahl und Caching helfen, die Ausgaben unter Kontrolle zu halten, ohne die Qualität zu opfern.


## Erstellen eines beobachtbaren Agenten

Wir definieren Reisewerkzeuge und versehen den Aufruf des Agenten mit einer Zeitmessung, sodass wir die Latenz überwachen können. In Produktionsumgebungen würden Sie OpenTelemetry oder ein ähnliches Tracing-Backend integrieren.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Bewertungsmuster

Ein verbreitetes Produktionsmuster ist, einen zweiten Agenten als **Evaluator** zu verwenden. Der Evaluator bewertet die Antwort des primären Agenten anhand vordefinierter Kriterien wie Vollständigkeit, Genauigkeit und Nützlichkeit.

Dies ermöglicht:
- Automatisierte Qualitätsprüfungen, bevor Antworten die Benutzer erreichen
- Erkennung von Regressionen, wenn sich Eingabeaufforderungen oder Modelle ändern
- Kontinuierliche Überwachung der Agentenleistung im Zeitverlauf


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategien zur Kostenkontrolle

Die Kostenkontrolle ist entscheidend für Produktions-KI-Agenten. Hier sind die wichtigsten Strategien:

| Strategie | Beschreibung |
|---|---|
| **Prompt-Optimierung** | Halten Sie Systemanweisungen kurz. Entfernen Sie redundanten Kontext, um die Anzahl der Eingabe-Token zu reduzieren. |
| **Modellauswahl** | Verwenden Sie kleinere, günstigere Modelle (z. B. GPT-4o-mini) für einfache Aufgaben wie Klassifizierung oder Extraktion und nutzen Sie größere Modelle nur für komplexes Schlussfolgern. |
| **Zwischenspeicherung** | Zwischenspeichern Sie Tool-Ergebnisse und häufige Abfragen, um redundante API-Aufrufe zu vermeiden. |
| **Token-Budgets** | Legen Sie Grenzwerte für `max_tokens` fest, um unerwartet lange Antworten zu verhindern. |
| **Bündelung** | Fassen Sie mehrere Benutzeranfragen, wo möglich, zu einem einzelnen API-Aufruf zusammen. |

In der Praxis funktioniert ein gestuftes Vorgehen gut: Leiten Sie einfache Anfragen an ein schnelles, kostengünstiges Modell weiter und eskalieren Sie nur komplexe Anfragen an ein leistungsfähigeres Modell.


## Zusammenfassung

In dieser Lektion haben Sie gelernt, wie Sie:

1. **Beobachtbarkeit hinzufügen** zu Interaktionen mit Agenten durch Zeitmessung und Protokollierung, und damit die Grundlage für Tracing und Monitoring schaffen.
2. **Agentenantworten bewerten** automatisch mithilfe eines Bewertungs-Agenten, der Vollständigkeit, Genauigkeit und Nützlichkeit bewertet.
3. **Kosten verwalten** durch Prompt-Optimierung, Modellauswahl, Caching und Token-Budgets.

Diese Produktionsmuster helfen sicherzustellen, dass Ihre KI-Agenten zuverlässig, messbar und kosteneffizient im großen Maßstab sind.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Haftungsausschluss:
Dieses Dokument wurde mithilfe des KI-Übersetzungsdienstes [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner ursprünglichen Sprache ist als maßgebliche Quelle zu betrachten. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
